# Fine-Tuning Object Detection Models on Custom Datasets

A comprehensive pipeline for training DETR-based object detection models on COCO-format datasets.

## 1. Environment Setup

### 1.1 Configuration

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class Config:
    """Training configuration."""
    
    # Model
    model_name: str = "microsoft/conditional-detr-resnet-50"
    image_size: int = 480
    
    # Dataset
    coco_dir: str = "/kaggle/input/zaloai25/coco_dataset"  # For local: "coco_dataset"
    
    # Training
    output_dir: str = "detr_finetuned_zalo_ai"
    num_epochs: int = 30
    batch_size: int = 32
    learning_rate: float = 5e-5
    weight_decay: float = 1e-4
    
    # Other
    push_to_hub: bool = False


config = Config()

### 1.2 GPUs

Note that you should use 1 GPU to avoid errors. Or use `os.environ["CUDA_VISIBLE_DEVICES"]="0"` to set only one GPU as visible.

### 1.3 WandB Integration

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb-det-zlai-25") # Replace with your secret name
wandb.login()

### 1.4 Dependencies

In [ ]:
!pip install -q datasets transformers[torch] accelerate timm
!pip install -q albumentations>=1.4.5 torchmetrics pycocotools
!pip install -q --upgrade numpy==1.26.4 scipy==1.12.0

### 1.5 Imports

In [ ]:
import json
import torch
import numpy as np
import albumentations as A
import matplotlib.pyplot as plt
from functools import partial
from pprint import pprint
from PIL import Image, ImageDraw
from pathlib import Path
from typing import Dict, List, Any

from datasets import Dataset, DatasetDict, Features, Value, Sequence, ClassLabel
from datasets import Image as HFImage
from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    TrainingArguments,
    Trainer,
)
from transformers.image_transforms import center_to_corners_format
from torchmetrics.detection.mean_ap import MeanAveragePrecision

## 2. Data Pipeline

### 2.1 Dataset Loading

**COCO Format Requirements:**
- Bounding boxes: `[x_min, y_min, width, height]` (absolute pixels)
- Categories: 0-indexed integers
- Valid bboxes: width > 0 and height > 0

In [ ]:
def load_coco_split(split_dir: Path) -> tuple[List[Dict], List[str]]:
    """Load COCO annotations and create HuggingFace dataset.
    
    Args:
        split_dir: Directory containing annotations/ and images/ folders
        
    Returns:
        Tuple of (dataset items, category names)
    """
    ann_file = split_dir / "annotations" / "instances.json"
    img_dir = split_dir / "images"
    
    with open(ann_file) as f:
        coco = json.load(f)
    
    # Validate and group annotations by image_id
    anns_by_img = {}
    invalid_count = 0
    
    for ann in coco["annotations"]:
        bbox = ann["bbox"]
        if len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0:
            anns_by_img.setdefault(ann["image_id"], []).append(ann)
        else:
            invalid_count += 1
    
    if invalid_count > 0:
        print(f"skipped {invalid_count} invalid bboxes in {split_dir.name}")
    
    # Build dataset items
    items = []
    for img in coco["images"]:
        img_anns = anns_by_img.get(img["id"], [])
        if not img_anns:
            continue
            
        items.append({
            "image_id": img["id"],
            "image": str(img_dir / img["file_name"]),
            "width": img["width"],
            "height": img["height"],
            "objects": {
                "id": [a["id"] for a in img_anns],
                "area": [a["area"] for a in img_anns],
                "bbox": [a["bbox"] for a in img_anns],
                "category": [a["category_id"] for a in img_anns],
            },
        })
    
    categories = [c["name"] for c in coco["categories"]]
    return items, categories


# Load all splits
coco_path = Path(config.coco_dir)
print("Loading COCO dataset...")

train_data, categories = load_coco_split(coco_path / "train")
val_data, _ = load_coco_split(coco_path / "val")
test_data, _ = load_coco_split(coco_path / "test")

# Define dataset features
features = Features({
    "image_id": Value("int64"),
    "image": HFImage(),
    "width": Value("int32"),
    "height": Value("int32"),
    "objects": {
        "id": Sequence(Value("int64")),
        "area": Sequence(Value("float32")),
        "bbox": Sequence(Sequence(Value("float32"), length=4)),
        "category": Sequence(ClassLabel(names=categories)),
    },
})

# Create datasets
dataset = DatasetDict({
    "train": Dataset.from_list(train_data, features=features),
    "validation": Dataset.from_list(val_data, features=features),
    "test": Dataset.from_list(test_data, features=features),
})

print(dataset)

### 2.2 Label Mapping

In [ ]:
# Create bidirectional label mappings
categories = dataset["train"].features["objects"]["category"].feature.names
id2label = {idx: cat for idx, cat in enumerate(categories)}
label2id = {cat: idx for idx, cat in enumerate(categories)}

print(f"Classes ({len(id2label)}): {list(id2label.values())}")

# Display sample
sample = dataset["train"][0]
print(f"\nSample: Image {sample['image_id']} | "
      f"Size {sample['width']}x{sample['height']} | "
      f"Objects {len(sample['objects']['id'])}")

### 2.3 Data Visualization

In [ ]:
def visualize_sample(dataset: Dataset, idx: int, id2label: Dict[int, str]) -> Image.Image:
    """Visualize a sample with bounding boxes.
    
    Args:
        dataset: Dataset to visualize from
        idx: Sample index
        id2label: Category ID to label mapping
        
    Returns:
        Annotated PIL Image
    """
    sample = dataset[idx]
    image = sample["image"].copy()
    draw = ImageDraw.Draw(image)
    annotations = sample["objects"]
    
    for i in range(len(annotations["id"])):
        x_min, y_min, width, height = annotations["bbox"][i]
        x_max, y_max = x_min + width, y_min + height
        category_id = annotations["category"][i]
        
        draw.rectangle((x_min, y_min, x_max, y_max), outline="red", width=2)
        draw.text((x_min, y_min - 10), id2label[category_id], fill="red")
    
    return image


# Visualize training samples
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, ax in enumerate(axes.flat):
    image = visualize_sample(dataset["train"], idx, id2label)
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(f"Sample {idx}")
plt.tight_layout()
plt.show()

### 2.4 Data Preprocessing

In [ ]:
# Initialize image processor
image_processor = AutoImageProcessor.from_pretrained(
    config.model_name,
    do_resize=True,
    size={"max_height": config.image_size, "max_width": config.image_size},
    do_pad=True,
    pad_size={"height": config.image_size, "width": config.image_size},
)

In [ ]:
# Define augmentation pipelines
train_augmentation = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(p=0.1),
        A.Perspective(p=0.1),
    ],
    bbox_params=A.BboxParams(
        format="coco",
        label_fields=["category"],
        clip=True,
    ),
)

validation_transform = A.Compose(
    [A.NoOp()],
    bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True),
)

In [ ]:
def format_annotations(
    image_id: int,
    categories: List[int],
    areas: List[float],
    bboxes: List[List[float]],
) -> Dict[str, Any]:
    """Format annotations to COCO format."""
    annotations = [
        {
            "image_id": image_id,
            "category_id": category,
            "iscrowd": 0,
            "area": area,
            "bbox": list(bbox),
        }
        for category, area, bbox in zip(categories, areas, bboxes)
    ]
    return {"image_id": image_id, "annotations": annotations}


def transform_batch(
    examples: Dict[str, List],
    transform: A.Compose,
    image_processor: AutoImageProcessor,
) -> Dict[str, torch.Tensor]:
    """Apply transformations to a batch of examples."""
    images = []
    annotations = []
    
    for image_id, image, objects in zip(
        examples["image_id"], examples["image"], examples["objects"]
    ):
        image = np.array(image.convert("RGB"))
        
        output = transform(
            image=image,
            bboxes=objects["bbox"],
            category=objects["category"],
        )
        
        if len(output["bboxes"]) == 0:
            continue
        
        images.append(output["image"])
        
        # Recompute areas after augmentation
        areas = [w * h for x, y, w, h in output["bboxes"]]
        
        annotations.append(
            format_annotations(image_id, output["category"], areas, output["bboxes"])
        )
    
    return image_processor(images=images, annotations=annotations, return_tensors="pt")


# Create transform functions
train_transform_fn = partial(
    transform_batch,
    transform=train_augmentation,
    image_processor=image_processor,
)

val_transform_fn = partial(
    transform_batch,
    transform=validation_transform,
    image_processor=image_processor,
)

# Apply transforms
train_dataset = dataset["train"].with_transform(train_transform_fn)
val_dataset = dataset["validation"].with_transform(val_transform_fn)
test_dataset = dataset["test"].with_transform(val_transform_fn)

print(f"Training: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")

### 2.5 Data Collator

In [ ]:
def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """Custom collate function to batch images and annotations."""
    valid_batch = [
        item for item in batch
        if isinstance(item, dict)
        and "pixel_values" in item
        and "labels" in item
        and item["pixel_values"].numel() > 0
        and item["labels"]
        and "boxes" in item["labels"]
        and len(item["labels"]["boxes"]) > 0
    ]
    
    if not valid_batch:
        return None
    
    data = {
        "pixel_values": torch.stack([x["pixel_values"] for x in valid_batch]),
        "labels": [x["labels"] for x in valid_batch],
    }
    
    if "pixel_mask" in valid_batch[0]:
        data["pixel_mask"] = torch.stack([x["pixel_mask"] for x in valid_batch])
    
    return data

## 3. Model Training

### 3.1 Evaluation Metrics

In [ ]:
@dataclass
class ModelOutput:
    """Wrapper for model outputs."""
    logits: torch.Tensor
    pred_boxes: torch.Tensor


def convert_bbox_yolo_to_pascal(
    boxes: torch.Tensor,
    image_size: tuple[int, int],
) -> torch.Tensor:
    """Convert bounding boxes from YOLO to Pascal VOC format."""
    boxes = center_to_corners_format(boxes)
    height, width = image_size
    boxes = boxes * torch.tensor([[width, height, width, height]])
    return boxes


@torch.no_grad()
def compute_metrics(
    evaluation_results,
    image_processor: AutoImageProcessor,
    threshold: float = 0.0,
    id2label: Dict[int, str] = None,
) -> Dict[str, float]:
    """Compute mAP and mAR metrics for object detection."""
    predictions, targets = evaluation_results.predictions, evaluation_results.label_ids
    
    # Prepare targets
    image_sizes = []
    post_processed_targets = []
    
    for batch in targets:
        batch_image_sizes = torch.tensor(np.array([x["orig_size"] for x in batch]))
        image_sizes.append(batch_image_sizes)
        
        for image_target in batch:
            boxes = convert_bbox_yolo_to_pascal(
                torch.tensor(image_target["boxes"]),
                image_target["orig_size"],
            )
            labels = torch.tensor(image_target["class_labels"])
            post_processed_targets.append({"boxes": boxes, "labels": labels})
    
    # Prepare predictions
    post_processed_predictions = []
    
    for batch, target_sizes in zip(predictions, image_sizes):
        batch_logits, batch_boxes = batch[1], batch[2]
        output = ModelOutput(
            logits=torch.tensor(batch_logits),
            pred_boxes=torch.tensor(batch_boxes),
        )
        post_processed_output = image_processor.post_process_object_detection(
            output, threshold=threshold, target_sizes=target_sizes
        )
        post_processed_predictions.extend(post_processed_output)
    
    # Compute metrics
    metric = MeanAveragePrecision(box_format="xyxy", class_metrics=True)
    metric.update(post_processed_predictions, post_processed_targets)
    metrics = metric.compute()
    
    # Format per-class metrics
    classes = metrics.pop("classes")
    map_per_class = metrics.pop("map_per_class")
    mar_100_per_class = metrics.pop("mar_100_per_class")
    
    for class_id, class_map, class_mar in zip(classes, map_per_class, mar_100_per_class):
        class_name = id2label[class_id.item()] if id2label else class_id.item()
        metrics[f"map_{class_name}"] = class_map
        metrics[f"mar_100_{class_name}"] = class_mar
    
    return {k: round(v.item(), 4) for k, v in metrics.items()}


# Create compute metrics function with preset parameters
compute_metrics_fn = partial(
    compute_metrics,
    image_processor=image_processor,
    id2label=id2label,
    threshold=0.0,
)

### 3.2 Model Initialization

In [ ]:
model = AutoModelForObjectDetection.from_pretrained(
    config.model_name,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

### 3.3 Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    learning_rate=config.learning_rate,
    lr_scheduler_type="cosine",
    weight_decay=config.weight_decay,
    max_grad_norm=0.01,
    fp16=True,
    dataloader_num_workers=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_map",
    greater_is_better=True,
    remove_unused_columns=False,
    eval_do_concat_batches=False,
    logging_steps=50,
    push_to_hub=config.push_to_hub,
)

### 3.4 Train Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=image_processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics_fn,
)

print("Starting training...")
trainer.train()

## 4. Evaluation & Inference

### 4.1 Test Set Evaluation

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
print("\n" + "="*50)
print("TEST RESULTS")
print("="*50)
pprint(test_metrics)

### 4.2 Save Model

In [ ]:
trainer.save_model(config.output_dir)
image_processor.save_pretrained(config.output_dir)
print(f"Model saved to {config.output_dir}")

if config.push_to_hub:
    trainer.push_to_hub()
    print("Model pushed to Hugging Face Hub")

### 4.3 Inference on Test Dataset

In [ ]:
def run_inference(
    image: Image.Image,
    model: AutoModelForObjectDetection,
    image_processor: AutoImageProcessor,
    threshold: float = 0.3,
) -> Dict[str, torch.Tensor]:
    """Run object detection inference on an image.
    
    Args:
        image: Input PIL Image
        model: Trained detection model
        image_processor: Image processor
        threshold: Confidence threshold
        
    Returns:
        Detection results with scores, labels, and boxes
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    inputs = image_processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]])
    results = image_processor.post_process_object_detection(
        outputs, threshold=threshold, target_sizes=target_sizes
    )[0]
    
    return results


def visualize_predictions(
    image: Image.Image,
    results: Dict[str, torch.Tensor],
    id2label: Dict[int, str],
    threshold: float = 0.3,
) -> Image.Image:
    """Visualize detection results on an image."""
    draw = ImageDraw.Draw(image)
    
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        if score > threshold:
            x1, y1, x2, y2 = [round(i, 2) for i in box.tolist()]
            
            draw.rectangle((x1, y1, x2, y2), outline="red", width=3)
            label_text = f"{id2label[label.item()]}: {score:.2f}"
            draw.text((x1, y1 - 10), label_text, fill="red")
    
    return image


# Select a random test image from test_dataset
test_idx = 0
test_sample = dataset["test"][test_idx]
test_image = test_sample["image"]

print(f"Running inference on test image {test_sample['image_id']}...")

# Run inference
results = run_inference(test_image, model, image_processor, threshold=0.3)

# Print detections
print(f"\nDetected {len(results['scores'])} objects:")
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box_str = [round(i, 2) for i in box.tolist()]
    label_name = model.config.id2label[label.item()]
    print(f"  {label_name}: {round(score.item(), 3)} | {box_str}")

In [ ]:
# Visualize predictions
annotated_image = visualize_predictions(
    test_image.copy(),
    results,
    model.config.id2label,
    threshold=0.3,
)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title(f"Object Detection Results - Test Image {test_sample['image_id']}")
plt.show()